In [1]:
from torch.utils.data import DataLoader
from datasets import load_from_disk
from src.data_utils import get_ravel_prefix_suffix_collate_fn

from transformers import AutoTokenizer

%load_ext autoreload
%autoreload 2

In [2]:
tokenizer = AutoTokenizer.from_pretrained("/work/frink/models/llama3-8B-HF")
tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

city_train_set = load_from_disk("./data/ravel/city_Latitude")["train"]
city_test_set = load_from_disk("./data/ravel/city_Latitude")["test"]

collate_fn = get_ravel_prefix_suffix_collate_fn(tokenizer, disentangling=True)

batch_size = 8  # 50 or so
data_loader = DataLoader(
    city_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=False
)  # batch_size, collate_fn=collate_fn)
test_data_loader = DataLoader(
    city_test_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=False
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
from src.llama3.model import RavelInterpretorHypernetwork


hypernetwork = RavelInterpretorHypernetwork(
    model_name_or_path="/work/frink/models/llama3-8B-HF",
    num_editing_heads=32,
    intervention_layer=15,
    das_intervention=True,
    das_dimension=256
)

hypernetwork = hypernetwork.to("cuda")
hypernetwork.load_model("./models/city_Latitude_das/model_epoch_9_step_298")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [15]:
hypernetwork.eval_accuracy(test_data_loader, disentangling=True, inference_mode=None)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


(0.384180790960452,
 2.3832023090786403,
 [0,
  4,
  6,
  8,
  12,
  14,
  19,
  20,
  22,
  25,
  27,
  28,
  29,
  33,
  36,
  41,
  45,
  46,
  47,
  48,
  57,
  61,
  65,
  77,
  78,
  80,
  83,
  86,
  88,
  89,
  93,
  96,
  97,
  99,
  101,
  102,
  103,
  104,
  105,
  106,
  109,
  112,
  114,
  118,
  122,
  125,
  126,
  128,
  129,
  130,
  131,
  133,
  134,
  137,
  138,
  140,
  142,
  143,
  145,
  148,
  151,
  154,
  162,
  163,
  168,
  171,
  175,
  184,
  185,
  187,
  188,
  193,
  194,
  195,
  198,
  205,
  206,
  211,
  213,
  214,
  215,
  216,
  218,
  220,
  221,
  222,
  229,
  231,
  232,
  233,
  234,
  235,
  238,
  239,
  240,
  241,
  242,
  243,
  244,
  246,
  247,
  252,
  257,
  263,
  264,
  272,
  276,
  277,
  284,
  285,
  288,
  290,
  294,
  296,
  301,
  302,
  304,
  308,
  310,
  312,
  314,
  316,
  318,
  319,
  320,
  322,
  327,
  329,
  334,
  337,
  340,
  342,
  346,
  349,
  350,
  89])

In [4]:
hypernetwork.interpretor.boundless_das.get_boundary_sparsity()

tensor(0.0625, device='cuda:0')

In [5]:
hypernetwork.plot_heatmap(test_data_loader, idxs=0, inference_mode=None, disentangling=True, annot=False)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


RuntimeError: a Tensor with 5 elements cannot be converted to Scalar